# Model the Paid Search return

*Data Analysis in Python · Topic 10.1*

Paid Search gets the most money of any channel, and your media team wants to know what one more dollar of it brings back. Fit the model on spend alone, then with a control for demand, and give them the return per dollar with its interval.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the clean `sessions` table (lesson 3.'s steps) and the cell that builds `campaign_days` (lesson 5.'s steps): spend and revenue per campaign and day, with each campaign's `channel`.
2. In the first cell marked ✏️, create the `weekly` table: one row per full Monday-to-Sunday week, with the dates as the index, and the columns:
   - `spend` and `revenue`: the Paid Search campaigns' totals, the rows of `campaign_days` whose `channel` is `search`
   - `organic`: the `sessions` table's sessions whose `session_medium` is `organic`
3. In the second cell marked ✏️, create `naive_model`: a regression of the `weekly` table's revenue on spend alone. Then create `naive_return`: its `spend` coefficient.
4. In the last cell marked ✏️, create `model`: a regression of revenue on spend and organic. Then create:
   - `spend_return`: its `spend` coefficient
   - `return_low` and `return_high`: the 95% confidence interval of that coefficient
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `weekly` table has 104 rows. `naive_return` is about `3.193`. `spend_return` is about `1.983`, with `return_low` about `1.595` and `return_high` about `2.371`.

## Parameters

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Spend and revenue per campaign and day, with lesson 5's steps, and each campaign's channel from its name. Run it, and leave it as it is.
def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google["spend"] = google["cost_micros"] / 1_000_000
spend = pd.concat([
    google.rename(columns={"campaign_name": "campaign"})[["date", "campaign", "spend"]],
    meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})[["date", "campaign", "spend"]],
    youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})[["date", "campaign", "spend"]],
], ignore_index=True)
spend["campaign"] = campaign_key(spend["campaign"])
spend = spend.groupby(["date", "campaign"], as_index=False)["spend"].sum()

revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])

campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
campaign_days["channel"] = campaign_days["campaign"].str.split("_").str[2]
campaign_days.shape

## One row per week

In [ ]:
# ✏️ Create the weekly table


## Spend alone

In [ ]:
# ✏️ Create naive_model and naive_return


## With a control for demand

In [ ]:
# ✏️ Create model, spend_return, return_low and return_high


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.0001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    weekly = found.get("weekly")
    if not isinstance(weekly, pd.DataFrame) or not {"spend", "revenue", "organic"} <= set(weekly.columns):
        results.append((False, "weekly should be a DataFrame with the columns spend, revenue and organic, one row per full week."))
    elif len(weekly) == 731:
        results.append((False, "The weekly table has 731 rows: one per day. Total the days by week with resample(\"W\"), as in topic 6.2."))
    elif len(weekly) == 105:
        results.append((False, "The weekly table has 105 rows: the last one is December 29 to 31, 2025, three days. Keep the weeks with 7 days, as in topic 6.2."))
    elif len(weekly) != 104:
        results.append((False, f"The weekly table should have the 104 full weeks of 2024 and 2025. It has {len(weekly):,} rows."))
    elif abs(weekly["spend"].sum() - 305812.10) > 0.05:
        total = weekly["spend"].sum()
        hint = " That's every channel's spend: keep the rows whose channel is search." if total > 500000 else ""
        results.append((False, f"The weekly table's spend adds up to ${total:,.2f}. The Paid Search campaigns' full weeks add up to $305,812.10.{hint}"))
    elif abs(weekly["revenue"].sum() - 728259.69) > 0.05:
        results.append((False, f"The weekly table's revenue adds up to ${weekly['revenue'].sum():,.2f}. The Paid Search campaigns' full weeks add up to $728,259.69: the revenue from the same campaign days as the spend."))
    elif weekly["organic"].sum() != 598953:
        results.append((False, f"The weekly table's organic column adds up to {weekly['organic'].sum():,}. It should be the sessions whose session_medium is organic, per week: 598,953."))
    else:
        results.append((True, f"104 full weeks: ${weekly['spend'].sum():,.2f} of Paid Search spend and ${weekly['revenue'].sum():,.2f} of revenue, ROAS {weekly['revenue'].sum() / weekly['spend'].sum():.2f}."))

    naive = found.get("naive_return")
    if not number(naive):
        results.append((False, f"naive_return should be a number: naive_model.params[\"spend\"]. It is {naive!r}."))
    elif near(naive, -2385.2403, tolerance=0.01):
        results.append((False, "naive_return is -2,385.24: that's the intercept. The spend coefficient is naive_model.params[\"spend\"]."))
    elif near(naive, 0.223899):
        results.append((False, "naive_return is 0.2239: that's spend explained by revenue. Put the outcome on the left: \"revenue ~ spend\"."))
    elif near(naive, 3.249429):
        results.append((False, "naive_return is 3.249: that's a model of days. Model full weeks."))
    elif near(naive, 3.103395):
        results.append((False, "naive_return is 3.103: that's every channel's spend. Keep the Paid Search campaigns."))
    elif not near(naive, 3.192564):
        results.append((False, f"naive_return should be the spend coefficient of revenue ~ spend on the weekly table: about 3.193. It is {naive:.4f}."))
    else:
        results.append((True, f"On spend alone, each extra dollar comes with ${naive:.2f} of revenue: more than the average dollar's $2.38. That's the warning sign."))

    spend_return, low, high = found.get("spend_return"), found.get("return_low"), found.get("return_high")
    if not number(spend_return):
        results.append((False, f"spend_return should be a number: model.params[\"spend\"], from revenue ~ spend + organic. It is {spend_return!r}."))
    elif near(spend_return, 3.192564):
        results.append((False, "spend_return is 3.193, the same as naive_return. Add organic to the formula: \"revenue ~ spend + organic\"."))
    elif near(spend_return, 0.839466):
        results.append((False, "spend_return is 0.839: that's the organic coefficient. Read model.params[\"spend\"]."))
    elif near(spend_return, 2.118431):
        results.append((False, "spend_return is 2.118: that's a model of days. Model full weeks."))
    elif near(spend_return, 1.786922):
        results.append((False, "spend_return is 1.787: the control is all sessions, and paid sessions come from the spend itself. Control for organic sessions, which follow demand without any spend behind them."))
    elif not near(spend_return, 1.983305):
        results.append((False, f"spend_return should be the spend coefficient of revenue ~ spend + organic: about 1.983. It is {spend_return:.4f}."))
    elif not number(low) or not number(high):
        results.append((False, "return_low and return_high should be numbers: model.conf_int().loc[\"spend\"], low and high."))
    elif near(low, 2.796521) and near(high, 3.588607):
        results.append((False, "return_low and return_high are naive_model's interval. Use the model with organic."))
    elif near(low, 1.658536) and near(high, 2.308074):
        results.append((False, "return_low and return_high are a 90% interval. The brief asks for 95%, conf_int()'s default."))
    elif not (near(low, 1.595219) and near(high, 2.371391)):
        results.append((False, f"return_low and return_high should be the 95% interval of the spend coefficient: about 1.595 and 2.371. They are {low:.4f} and {high:.4f}."))
    else:
        results.append((True, f"With demand held constant, each extra dollar comes with ${spend_return:.2f}, somewhere from ${low:.2f} to ${high:.2f}: below the average dollar's $2.38, as the next dollar should be."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
search_daily = campaign_days[campaign_days["channel"] == "search"].groupby("date")[["spend", "revenue"]].sum()
search_daily["organic"] = sessions[sessions["session_medium"] == "organic"].groupby("date")["sessions"].sum()
days = search_daily["spend"].resample("W").count()
weekly = search_daily.resample("W").sum()
weekly = weekly[days == 7]
weekly.shape
```

```python
naive_model = smf.ols("revenue ~ spend", data=weekly).fit()
naive_return = naive_model.params["spend"]
naive_return, naive_model.rsquared
```

```python
model = smf.ols("revenue ~ spend + organic", data=weekly).fit()
spend_return = model.params["spend"]
return_low, return_high = model.conf_int().loc["spend"]
spend_return, return_low, return_high, model.rsquared
```

</details>